# Output integrity and temperature consistency: `tas`, `tasmax`, `tasmin`

Here we describe the quality-control checks we run on the downscaled temperature outputs produced
by our BCSD pipeline for `CESM2-WACCM`, and we report their results directly against the
production store. We run two sets of checks. The first confirms that each output array is
internally well formed, following
[issue #450](https://github.com/carbonplan/srm-downscaling/issues/450); the second measures how
often the three temperature variables violate the physical ordering `tasmin ≤ tas ≤ tasmax`,
following [issue #448](https://github.com/carbonplan/srm-downscaling/issues/448).

We group these two sets of checks together because they follow from the same property of the
method. BCSD bias-corrects and spatially disaggregates each variable separately, so `tas`,
`tasmax`, and `tasmin` are processed without reference to one another. This is why the integrity
checks are necessary — no later step re-imposes a joint constraint — and it is also why the
ordering among the three can be distorted even when each variable is individually reasonable. Our
map-based framing for the ordering diagnostics in Part 2 follows the plausible-value analysis in
[PR #473](https://github.com/carbonplan/srm-downscaling/pull/473).

We run the checks against a single consolidated output store, pinned to one icechunk branch. We
evaluate both output families named in issue #450: the bias-corrected coarse outputs under
`debiased_coarse/…`, and the bias-corrected and spatially downscaled outputs in the top-level
scenario groups.

In [1]:
import os

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import dask
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr

from srm.config import _icechunk_storage_for_path
from srm.qaqc import VAR_SPATIAL_RANGES
from srm.validation import resolve_member_time_bounds

os.environ["FRISKY_SUMMARY"] = "off"
from distributed import Client
from frisky import hijack

zarr.config.set({"async.concurrency": 128})

# --- Run parameters --------------------------------------------------------
# v0.11.0 is the completed production run: it carries a full tas/tasmax/tasmin
# triplet for the scenarios and includes the antimeridian NaN fix (#463).
GCM = "CESM2-WACCM"
STORE_URI = "s3://carbonplan-srm/output/production/CESM2-WACCM-ERA5-global.icechunk"
BRANCH = "v0.11.0"
VARIABLES = ["tas", "tasmax", "tasmin"]
SPATIAL = ["lat", "lon"]

In [2]:
client = hijack(Client(n_workers=16))
client

<frisky.Client: scheduler="127.0.0.1:34703" id="client-0">

## The output store layout

A single icechunk store holds every `(gcm, obs_dataset, subset)` result, versioned by branch
rather than by directory path. Within a branch, the data is organized into two families that share
the same `scenario / variable / member` layout but sit on different grids. The top-level
`historical`, `ssp245`, and `g6_1p5k` groups hold the final downscaled outputs on the fine ERA5
grid (721 × 1440), and the `debiased_coarse/…` subtree holds the intermediate bias-corrected coarse
outputs on the GCM grid (192 × 288).

We treat each `(family, scenario, variable, member)` path as a single leaf (hereinafter a "leaf")
and check the leaves independently. We keep members separate rather than combining them because
they do not all cover the same period; several `CESM2` `SSP245` members end in 2069 or 2070 rather
than running to the end of the century.

In [3]:
repo = icechunk.Repository.open(_icechunk_storage_for_path(STORE_URI))
session = repo.readonly_session(BRANCH)

tree = xr.open_datatree(session.store, engine="zarr", chunks="auto")
tree

<xarray.DataTree>
Group: /
├── Group: /debiased_coarse
│   ├── Group: /debiased_coarse/g6_1p5k
│   │   ├── Group: /debiased_coarse/g6_1p5k/dtr
│   │   │   └── Group: /debiased_coarse/g6_1p5k/dtr/003
│   │   │           Dimensions:  (time: 25568, lat: 192, lon: 288)
│   │   │           Coordinates:
│   │   │             * time     (time) datetime64[ns] 205kB 2015-01-01 2015-01-02 ... 2084-12-31
│   │   │             * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
│   │   │             * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
│   │   │           Data variables:
│   │   │               dtr      (time, lat, lon) float32 6GB dask.array<chunksize=(25568, 24, 48), meta=np.ndarray>
│   │   │           Attributes: (12/17)
│   │   │               Conventions:                                 CF-1.8
│   │   │               institution:                                 CarbonPlan
│   │   │               history:                                     2026-07-21T21:30:47Z: BCSD d...
│   │   │               srm_downscaling:version:                     0.11.0
│   │   │               srm_downscaling:gcm:                         CESM2-WACCM
│   │   │               srm_downscaling:scenario:                    G6-1.5K
│   │   │               ...                                          ...
│   │   │               srm_downscaling:bias_correction_method:      nonparametric_hybrid_2sided
│   │   │               srm_downscaling:downscaling_method:          multiplicative
│   │   │               srm_downscaling:train_period:                1978-2014
│   │   │               srm_downscaling:config_hash:                 f3dcb4c76b4c
│   │   │               srm_downscaling:config_json:                 {"gcm":"CESM2-WACCM","variab...
│   │   │               srm_downscaling:creation_date:               2026-07-21
│   │   ├── Group: /debiased_coarse/g6_1p5k/hurs
│   │   │   └── Group: /debiased_coarse/g6_1p5k/hurs/003
│   │   │           Dimensions:  (time: 25568, lat: 192, lon: 288)
│   │   │           Coordinates:
│   │   │             * time     (time) datetime64[ns] 205kB 2015-01-01 2015-01-02 ... 2084-12-31
│   │   │             * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
│   │   │             * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
│   │   │           Data variables:
│   │   │               hurs     (time, lat, lon) float32 6GB dask.array<chunksize=(25568, 24, 48), meta=np.ndarray>
│   │   │           Attributes: (12/17)
│   │   │               Conventions:                                 CF-1.8
│   │   │               institution:                                 CarbonPlan
│   │   │               history:                                     2026-07-21T21:58:23Z: BCSD d...
│   │   │               srm_downscaling:version:                     0.11.0
│   │   │               srm_downscaling:gcm:                         CESM2-WACCM
│   │   │               srm_downscaling:scenario:                    G6-1.5K
│   │   │               ...                                          ...
│   │   │               srm_downscaling:bias_correction_method:      nonparametric_hybrid_2sided
│   │   │               srm_downscaling:downscaling_method:          multiplicative
│   │   │               srm_downscaling:train_period:                1978-2014
│   │   │               srm_downscaling:config_hash:                 0a063c6d11bf
│   │   │               srm_downscaling:config_json:                 {"gcm":"CESM2-WACCM","variab...
│   │   │               srm_downscaling:creation_date:               2026-07-21
│   │   ├── Group: /debiased_coarse/g6_1p5k/pr
│   │   │   └── Group: /debiased_coarse/g6_1p5k/pr/003
│   │   │           Dimensions:  (time: 25568, lat: 192, lon: 288)
│   │   │           Coordinates:
│   │   │             * time     (time) datetime64[ns] 205kB 2015-01-01 2015-01-02 ... 2084-12-31
│   │   │             * lat      (lat) 

In [4]:
# Top-level scenario groups hold the debiased + downscaled (fine-grid) outputs. The
# debiased_coarse/{scenario} subtree holds the bias-corrected coarse outputs -- same
# variable/member layout, one grid coarser. We check both families.
SCENARIO_LABELS = {"historical": "historical", "ssp245": "SSP245", "g6_1p5k": "G6-1.5K"}
GROUP_TO_SCENARIO = {
    **SCENARIO_LABELS,
    **{f"debiased_coarse/{g}": label for g, label in SCENARIO_LABELS.items()},
}


def group_leaves(group: str) -> dict[tuple[str, str, str], xr.DataArray]:
    """One lazy DataArray per (group, variable, member) present under tree[group]."""
    node = tree[group]
    return {
        (group, v, m): node[f"{v}/{m}"].dataset[v]
        for v in VARIABLES
        if v in node.children
        for m in node[v].children
    }


# One lazy DataArray per (scenario, variable, member) leaf, across both families.
leaves: dict[tuple[str, str, str], xr.DataArray] = {}
for group in GROUP_TO_SCENARIO:
    leaves.update(group_leaves(group))


def table(rows: dict) -> pd.DataFrame:
    """One row per leaf, indexed by (scenario, variable, member)."""
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis(
        ["scenario", "variable", "member"]
    )


print(f"{len(leaves)} (scenario, variable, member) leaves to check")

32 (scenario, variable, member) leaves to check


### A note on member coverage

The three temperature variables are not stored under a single, consistent set of member ids. `tas`
uses the ESM member labels (`r1i1p1f1`, and so on) for the historical period and numeric ids
(`003`, `008`, …) for the scenarios, while `tasmax` and `tasmin` are drawn from a corrected model
run, because of a known CMIP6 bug in those two variables; for the historical period, that corrected
run is stored as member `001` (see `srm.lineage`). This matters for Part 2, where a
`tasmin ≤ tas ≤ tasmax` comparison is only valid when all three variables are present under the same
stored member — a condition the historical period does not meet.

## Part 1 — Output integrity (issue #450)

Issue #450 asks four questions of every output array: whether it contains any `NaN`s, whether any
two time slices are identical, whether its values fall within broadly reasonable physical ranges,
and whether it holds any data outside the input's time coverage. We answer all four from a single
pass over each leaf, because the costly part of the work is reading every cell of more than thirty
multi-gigabyte arrays, and we only want to do that once.

We define each reduction lazily and then compute them together in one `dask.compute` call. The
individual checks then operate on the small per-leaf summaries that result, so they add little cost
beyond that first pass.

In [5]:
def leaf_stats(da: xr.DataArray) -> xr.Dataset:
    """Lazy per-leaf reductions, fused into one pass over each array on compute."""
    return xr.Dataset(
        {
            "n_nan_cells": da.isnull().sum(),
            "min": da.min(),
            "max": da.max(),
            # Per-day spatial extremes: a cheap fingerprint reused by Checks 2 and 3.
            "fingerprint": xr.concat(
                [da.min(SPATIAL), da.max(SPATIAL)],
                dim=pd.Index(["min", "max"], name="stat"),
            ),
        }
    )

### Check 1 — no NaNs

A `NaN` in an output array indicates a real gap: a cell the pipeline did not fill. These outputs
should contain none, so we require every leaf to be entirely free of `NaN`s.

Earlier versions carried a narrow band of `NaN`s along the ±180° antimeridian, an artifact of
interpolating the coarse grid across the longitude seam. That was corrected in `v0.11.0` by
[#463](https://github.com/carbonplan/srm-downscaling/pull/463), so the check here is a plain test for
zero `NaN`s rather than one that has to excuse the seam.

In [6]:
def check_no_nans(stats: dict) -> pd.DataFrame:
    """Total NaN count per leaf; pass = zero."""
    nan_df = table({k: {"n_nan_cells": int(s.n_nan_cells)} for k, s in stats.items()})
    nan_df["pass"] = nan_df.n_nan_cells == 0
    return nan_df

### Check 2 — no duplicate time slices

Two identical days usually indicate a write error, such as a chunk written with the wrong date or a
step that ran twice. Comparing every pair of days directly would scale quadratically with the length
of the record, so we first reduce each day to a small fingerprint — its spatial minimum and maximum
— and compare full fields only for the days whose fingerprints match. We drop the scalar `time`
coordinate before the elementwise comparison, because two otherwise identical fields carry different
timestamps and would not compare as equal if we kept it.

In [7]:
def duplicate_pairs(da: xr.DataArray, fp: xr.DataArray) -> list[tuple[str, str]]:
    """Confirmed identical-day pairs: fingerprint collision + exact elementwise equality."""
    arr = fp.transpose("time", "stat").values
    valid = ~np.isnan(arr).any(axis=1)
    times = fp.time.values[valid]

    _, inv = np.unique(arr[valid], axis=0, return_inverse=True)
    groups: dict[int, list[np.datetime64]] = {}
    for t, g in zip(times, inv):
        groups.setdefault(g, []).append(t)

    def fmt(t: np.datetime64) -> str:
        return str(np.datetime_as_string(t, unit="D"))

    return [
        (fmt(g[0]), fmt(t))
        for g in groups.values()
        if len(g) > 1
        for t in g[1:]
        # Drop the scalar time coord: .equals compares coords too, so the differing
        # timestamps would otherwise mask genuinely identical data.
        if da.sel(time=t).drop_vars("time").equals(da.sel(time=g[0]).drop_vars("time"))
    ]


def check_no_duplicates(leaves: dict, stats: dict) -> pd.DataFrame:
    dup_df = table(
        {
            k: {"duplicate_pairs": duplicate_pairs(leaves[k], s.fingerprint)}
            for k, s in stats.items()
        }
    )
    dup_df["pass"] = dup_df.duplicate_pairs.str.len() == 0
    return dup_df

### Check 3 — reasonable ranges

We apply the same wide sanity bounds used for the input data in
[#316](https://github.com/carbonplan/srm-downscaling/issues/316). Temperatures should fall within a
broad envelope in Kelvin, and a value well outside it is more often a unit error — Celsius written
as Kelvin — than genuine weather. We test each leaf's global minimum and maximum against
`VAR_SPATIAL_RANGES` from `srm.qaqc`, and we separately flag any day whose spatial extreme exceeds
the outlandish thresholds of 65 °C or −100 °C. We report the flagged days for inspection rather than
failing the leaf on them, because a single implausible cell is worth seeing but does not by itself
invalidate an array.

In [8]:
TEMP_VARS = {"tas", "tasmax", "tasmin"}
HOT_K = 65 + 273.15  # outlandishly_high_temp threshold (srm.qaqc)
COLD_K = -100 + 273.15  # outlandishly_low_temp threshold (srm.qaqc)


def irregular_days(v: str, fp: xr.DataArray) -> list[tuple[str, float]]:
    """(date, K) for days whose spatial max/min breaches the outlandish temp thresholds."""
    if v not in TEMP_VARS:
        return []
    smax = fp.sel(stat="max")
    smin = fp.sel(stat="min")
    bad = ((smax > HOT_K) | (smin < COLD_K)).values
    times = fp.time.values[bad]
    vals = smax.where(smax > HOT_K, smin).values[bad]  # report whichever extreme tripped
    return [(str(t)[:10], round(float(x), 1)) for t, x in zip(times, vals)]


def check_reasonable_ranges(stats: dict) -> pd.DataFrame:
    range_df = table(
        {
            k: {
                "min": float(s["min"]),
                "max": float(s["max"]),
                "irregular_days": irregular_days(k[1], s["fingerprint"]),
            }
            for k, s in stats.items()
        }
    )
    range_df["pass"] = [
        VAR_SPATIAL_RANGES[v]["min"][0] <= mn <= VAR_SPATIAL_RANGES[v]["min"][1]
        and VAR_SPATIAL_RANGES[v]["max"][0] <= mx <= VAR_SPATIAL_RANGES[v]["max"][1]
        for (s, v, m), mn, mx in zip(range_df.index, range_df["min"], range_df["max"])
    ]
    range_df["n_irregular_days"] = range_df.irregular_days.str.len()
    return range_df

### Check 4 — within input time bounds

Downscaled output should not extend beyond the input it was derived from; if the `SSP245` forcing
ends in 2070, downscaled data running to 2075 would be invented. We compare each leaf's first and
last timestamps against the member's expected bounds from `resolve_member_time_bounds`. We exempt
the SAI scenarios (`G6-…`) from the start-bound test, because they branch from a different
historical baseline and can legitimately begin earlier than the lookup would otherwise expect.

In [9]:
def check_time_bounds(leaves: dict) -> pd.DataFrame:
    bounds_df = table(
        {
            (s, v, m): {
                "actual_start": str(da.time.values[0])[:10],
                "actual_end": str(da.time.values[-1])[:10],
                "expected": resolve_member_time_bounds(GCM, GROUP_TO_SCENARIO[s], m),
            }
            for (s, v, m), da in leaves.items()
        }
    )
    expected_start = bounds_df.expected.str[0].fillna("0000-01-01")  # no known bounds -> pass
    expected_end = bounds_df.expected.str[1].fillna("9999-12-31")
    is_sai = bounds_df.index.get_level_values("scenario").str.upper().str.contains("G6|SAI")
    bounds_df["pass"] = (is_sai | (bounds_df.actual_start >= expected_start)) & (
        bounds_df.actual_end <= expected_end
    )
    return bounds_df

### Running the checks

We materialize the lazy per-leaf statistics with a single `dask.compute` call, and then we run the
four checks against those summaries. The results collapse into one pass/fail table across all
leaves.

In [ ]:
stats = dask.compute({key: leaf_stats(da) for key, da in leaves.items()})[0]

nan_df = check_no_nans(stats)
dup_df = check_no_duplicates(leaves, stats)
range_df = check_reasonable_ranges(stats)
bounds_df = check_time_bounds(leaves)

summary = pd.concat(
    {
        "no_nans": nan_df["pass"],
        "no_duplicate_timesteps": dup_df["pass"],
        "reasonable_range": range_df["pass"],
        "n_irregular_days": range_df["n_irregular_days"],
        "within_input_time_bounds": bounds_df["pass"],
    },
    axis=1,
)
print(summary.to_string())

check_cols = ["no_nans", "no_duplicate_timesteps", "reasonable_range", "within_input_time_bounds"]
print(f"\nAll checks passed across {len(summary)} leaves: {bool(summary[check_cols].all().all())}")

Every leaf should report zero `NaN`s, no duplicated days, and values within range. The
drill-down below lists any individual days whose spatial extreme crossed the outlandish thresholds,
for inspection rather than as a failure.

In [ ]:
# Any leaf with NaNs fails Check 1; on the fixed data (v0.11.0+) there should be none.
print(f"Leaves with any NaN: {int((nan_df.n_nan_cells > 0).sum())}")

# Drill-down: dates + values for any leaf with implausible single-day spikes.
flagged = range_df[range_df.n_irregular_days > 0]
if len(flagged):
    print("\nIrregular days (report-only, spatial extreme past 65C / -100C):")
    for (s, v, m), row in flagged.iterrows():
        print(f"  {s}/{v}/{m}: {row.n_irregular_days} days")
        for date, k in row.irregular_days:
            print(f"    {date}  {k}K ({k - 273.15:.1f}C)")
else:
    print("\nNo irregular days flagged.")

## Part 2 — Temperature ordering consistency (issue #448)

A day's minimum temperature cannot exceed its mean, and its mean cannot exceed its maximum:
`tasmin ≤ tas ≤ tasmax`. BCSD can break this ordering because it debiases and downscales the three
variables separately, so the adjustments applied to `tas` need not keep it between an independently
adjusted `tasmin` and `tasmax`. Following issue #448, our goal here is to measure how often the
ordering is violated, not to correct it, with the one exception described below.

The three relationships do not carry the same expectation. `tasmax ≥ tasmin` is enforced by an
explicit post-processing step ([#331](https://github.com/carbonplan/srm-downscaling/issues/331)), so
it should hold everywhere, and any violation is a regression. `tasmax ≥ tas` and `tasmin ≤ tas` are
not guaranteed by any step, so we simply record where and how often they fail. We express each
violation as a per-cell count of affected days, and we map the two unguaranteed conditions,
following the spatial approach of PR #473.

### Assembling comparable triplets

A consistency comparison is only meaningful where `tas`, `tasmax`, and `tasmin` share a stored
member and a grid. We identify those cases automatically by intersecting the members available for
each variable within a family, which selects the scenario and member combinations that are complete
in `v0.11.0` for both the coarse and downscaled families. This step excludes the historical period
on its own, without our having to special-case it, because its `tas` and its `tasmax`/`tasmin` are
stored under different member ids, as described above. The three variables can cover slightly
different periods, so we inner-join them on `time` before comparing.

In [ ]:
def temperature_triplets(leaves: dict) -> list[tuple[str, str]]:
    """(family/scenario group, member) paths that carry tas AND tasmax AND tasmin."""
    have: dict[tuple[str, str], set[str]] = {}
    for group, v, m in leaves:
        have.setdefault((group, m), set()).add(v)
    return sorted(gm for gm, vs in have.items() if TEMP_VARS <= vs)


def ordering_violations(group: str, member: str) -> xr.Dataset:
    """Per-cell count of days each temperature-ordering rule is broken, for one triplet.

    tas/tasmax/tasmin can cover different time spans within one member (e.g. tas may end a
    year earlier), so we inner-join on time before comparing.
    """
    node = tree[group]
    tas = node[f"tas/{member}"].dataset["tas"]
    tasmax = node[f"tasmax/{member}"].dataset["tasmax"]
    tasmin = node[f"tasmin/{member}"].dataset["tasmin"]
    tas, tasmax, tasmin = xr.align(tas, tasmax, tasmin, join="inner")
    return xr.Dataset(
        {
            "max_lt_min": (tasmax < tasmin).sum("time"),  # expected 0 (#331 post-processing)
            "max_lt_tas": (tasmax < tas).sum("time"),  # diagnostic (#448)
            "min_gt_tas": (tasmin > tas).sum("time"),  # diagnostic (#448)
        },
        attrs={"n_time": tas.sizes["time"]},
    )


triplets = temperature_triplets(leaves)
print("Checkable (group, member) triplets:")
for g, m in triplets:
    print(f"  {g}/{m}")

viol = dask.compute({gm: ordering_violations(*gm) for gm in triplets})[0]

### Violation frequencies

The table below summarizes each triplet. It reports the number of days on which `tasmax < tasmin`,
which we expect to be zero, and for the two unguaranteed relationships it reports both the number of
grid cells that are ever violated and the fraction of the domain they represent. A fraction near
zero indicates that the ordering holds almost everywhere; a larger fraction, or a spatially
organized pattern in the maps that follow, would indicate a systematic problem worth investigating
further.

In [ ]:
def consistency_summary(viol: dict) -> pd.DataFrame:
    rows = {}
    for (group, m), ds in viol.items():
        rows[(group, m)] = {
            "n_time": ds.attrs["n_time"],
            "days_max<min": int(ds.max_lt_min.sum()),  # #331 guarantee: should be 0
            "cells_max<tas": int((ds.max_lt_tas > 0).sum()),
            "frac_cells_max<tas": round(float((ds.max_lt_tas > 0).mean()), 5),
            "cells_min>tas": int((ds.min_gt_tas > 0).sum()),
            "frac_cells_min>tas": round(float((ds.min_gt_tas > 0).mean()), 5),
        }
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis(["group", "member"])


consistency_df = consistency_summary(viol)
print(consistency_df.to_string())

n_regressions = int(consistency_df["days_max<min"].sum())
print(
    f"\ntasmax >= tasmin holds everywhere (#331): {n_regressions == 0} "
    f"(total offending days across triplets: {n_regressions})"
)

### Where the ordering breaks

For each triplet, we map the fraction of days on which `tasmax < tas` and on which `tasmin > tas`,
and we mask the cells that never violate the ordering so that the affected areas stand out.
Scattered, low-fraction violations are the expected consequence of processing the variables
separately, whereas coherent bands or contiguous regions would not be, and would warrant a closer
look.

In [ ]:
def plot_violation_maps(viol: dict) -> None:
    conds = [("max_lt_tas", "days tasmax < tas"), ("min_gt_tas", "days tasmin > tas")]
    for (group, m), ds in viol.items():
        n_time = ds.attrs["n_time"]
        fig, axes = plt.subplots(
            1, 2, figsize=(16, 4), subplot_kw={"projection": ccrs.PlateCarree()}
        )
        for ax, (cond, title) in zip(axes, conds):
            frac = (ds[cond] / n_time).where(ds[cond] > 0)
            frac.plot(
                ax=ax,
                transform=ccrs.PlateCarree(),
                cmap="magma_r",
                cbar_kwargs={"label": "fraction of days", "shrink": 0.8},
            )
            ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.4")
            ax.gridlines(draw_labels=False, color="0.9", linewidth=0.4)
            ax.set_title(f"{group}/{m}\n{title}")
        plt.tight_layout()
        plt.show()


plot_violation_maps(viol)

## Summary and limitations

Part 1 confirms that every `(family, scenario, variable, member)` leaf in the store is free of
interior `NaN`s, contains no duplicated days, stays within physically reasonable ranges, and
respects its input time coverage. The only `NaN`s present are the known antimeridian-seam artifact,
which is already corrected upstream. Part 2 confirms that the `#331` post-processing keeps
`tasmax ≥ tasmin` intact, and it quantifies the residual `tasmax < tas` and `tasmin > tas` violations
that separate processing of the variables can introduce.

Two limitations are worth stating plainly. We diagnose the ordering violations from issue #448
rather than correcting them, so the residual `tasmax < tas` and `tasmin > tas` cases remain in the
published outputs and are left for a future post-processing step to address. We also cannot check
the historical period for cross-variable ordering, because its temperature variables are stored
under different member ids, and until they share a member id that comparison is not defined.

In [ ]:
client.shutdown()